# Chapter 5, Exercise 1: Enumerating CTC paths

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 5, Exercise 1.** Using Python, write a short function that generates all fifteen valid length-4 CTC frame labelings that collapse to the 2-token target 'A B' (writing blank as a dash) and prints them. Then extend it to print one invalid length-4 labeling, with what it collapses to and why it fails.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


## 1. The CTC collapse rule

A CTC path assigns one symbol (a token or the blank `-`) to every encoder time step. The path becomes a transcript in two steps (Section 5.2): first merge consecutive repetitions of the same token, then remove the blanks. We implement exactly that.

In [1]:
from itertools import product

def ctc_collapse(path, blank="-"):
    """Merge consecutive repeats, then drop blanks."""
    merged = []
    for sym in path:
        if not merged or sym != merged[-1]:
            merged.append(sym)
    return tuple(s for s in merged if s != blank)

def valid_paths(target, T, alphabet=("A", "B"), blank="-"):
    """All length-T labelings over alphabet + blank that collapse to `target`."""
    target = tuple(target)
    return [p for p in product(list(alphabet) + [blank], repeat=T) if ctc_collapse(p, blank) == target]

paths = valid_paths(("A", "B"), T=4)
print(f"{len(paths)} valid length-4 paths collapse to 'A B':")
for p in paths:
    print("  ", " ".join(p))

15 valid length-4 paths collapse to 'A B':
   A A A B
   A A B B
   A A B -
   A A - B
   A B B B
   A B B -
   A B - -
   A - B B
   A - B -
   A - - B
   - A A B
   - A B B
   - A B -
   - A - B
   - - A B


There are exactly **15** valid paths, matching the worked example in Section 5.2. During training CTC sums the probabilities of these 15 paths to obtain P('A B' | audio); it never needs to be told which one is "correct".

*Why 15?* A valid path has the shape `blanks* A+ blanks* B+ blanks*`: five slots whose lengths sum to 4 with the two token runs at least 1 long. Subtracting the mandatory A and B leaves 2 free positions to distribute over 5 slots, C(2+4, 4) = 15.

## 2. An invalid labeling, and why it fails

In [2]:
def explain(path, target=("A", "B")):
    out = ctc_collapse(path)
    status = "VALID" if out == target else "INVALID"
    print(f"{' '.join(path)}  ->  collapses to {' '.join(out) or '(empty)'}   [{status}]")
    return out

# 1. Two consecutive A's after a B: the order is wrong
explain(("B", "A", "A", "-"))
# 2. Blank between the B's does not matter, but a blank between A and A is fatal for a single A
explain(("A", "-", "A", "B"))
# 3. Only the first token appears
explain(("A", "A", "-", "-"))
# 4. Repeated tokens with no blank merge into one: a geminate needs a blank!
explain(("A", "B", "B", "B"))   # this one IS valid: repeated B's merge into one B

B A A -  ->  collapses to B A   [INVALID]
A - A B  ->  collapses to A A B   [INVALID]
A A - -  ->  collapses to A   [INVALID]
A B B B  ->  collapses to A B   [VALID]


('A', 'B')

Reading the four cases:

* `B A A -` collapses to `B A`. CTC paths are **monotonic**: tokens must appear in target order, and no path can produce `A B` from a `B` that comes first.
* `A - A B` collapses to `A A B`. A blank between two identical tokens **prevents** their merger, so this path spells a target with two A's, not one. This is the mechanism the book highlights for Arabic gemination (Figure 5.1, ممكن): if a geminate is written as a repeated token, the path *must* contain a blank between the two copies; if it is written with a shadda token instead, no blank is needed.
* `A A - -` collapses to `A`; the second token was never emitted.
* `A B B B` is valid: consecutive repeats of B merge into one B. Many paths, one transcript, which is exactly what the sum in the CTC loss exploits.

## 3. Count valid paths for other lengths (optional)

In [3]:
for T in range(2, 8):
    print(f"T={T}: {len(valid_paths(('A','B'), T)):3d} paths for 'A B',   "
          f"{len(valid_paths(('A','A'), T)):3d} paths for 'A A' (needs a blank between the A's)")

T=2:   1 paths for 'A B',     0 paths for 'A A' (needs a blank between the A's)
T=3:   5 paths for 'A B',     1 paths for 'A A' (needs a blank between the A's)
T=4:  15 paths for 'A B',     5 paths for 'A A' (needs a blank between the A's)
T=5:  35 paths for 'A B',    15 paths for 'A A' (needs a blank between the A's)
T=6:  70 paths for 'A B',    35 paths for 'A A' (needs a blank between the A's)
T=7: 126 paths for 'A B',    70 paths for 'A A' (needs a blank between the A's)


The 'A A' column is the geminate case: for T = 2 there is **no** valid path at all (`A A` merges to one A), so a CTC model whose output writes a doubled consonant as two identical tokens needs at least three frames to emit it.